In [76]:
#codelab using sentence transformers - fiass to store embeddings

In [77]:
!pip install -q sentence-transformers faiss-cpu

In [78]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import faiss

In [79]:
#load pretrained model

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic Similarity (undersatnding meaning)

In [80]:
sentences = [
    "The cat sits outside",
    "A dog is playing in the garden",
    "A feline is sitting outdoorss",
    "The stock market crashed yesterday",
]

In [81]:
#generate embeddings

embeddings = model.encode(sentences, convert_to_tensor=True)

In [82]:
embeddings

tensor([[ 0.1392,  0.0030,  0.0470,  ...,  0.0641, -0.0163,  0.0636],
        [ 0.0139, -0.0803,  0.0410,  ...,  0.0360,  0.0278,  0.0035],
        [ 0.0981, -0.0101,  0.0706,  ...,  0.0355, -0.0239,  0.0114],
        [ 0.0640, -0.0425,  0.0397,  ..., -0.0691, -0.1011,  0.0745]])

In [83]:
embeddings.shape

torch.Size([4, 384])

In [84]:
cosine_scores = util.cos_sim(embeddings, embeddings)

In [85]:
cosine_scores

tensor([[1.0000e+00, 2.5725e-01, 6.7430e-01, 1.2480e-01],
        [2.5725e-01, 1.0000e+00, 3.2059e-01, 4.1975e-04],
        [6.7430e-01, 3.2059e-01, 1.0000e+00, 9.5453e-02],
        [1.2480e-01, 4.1975e-04, 9.5453e-02, 1.0000e+00]])

In [86]:
#semantic embeddings
for i in range(len(sentences)):
  for j in range(i+1, len(sentences)):
    print(f"{sentences[i]} \t\t {sentences[j]} \t\t Score: {cosine_scores[i][j]}")

The cat sits outside 		 A dog is playing in the garden 		 Score: 0.2572491466999054
The cat sits outside 		 A feline is sitting outdoorss 		 Score: 0.6743031144142151
The cat sits outside 		 The stock market crashed yesterday 		 Score: 0.12480233609676361
A dog is playing in the garden 		 A feline is sitting outdoorss 		 Score: 0.32059091329574585
A dog is playing in the garden 		 The stock market crashed yesterday 		 Score: 0.0004197489470243454
A feline is sitting outdoorss 		 The stock market crashed yesterday 		 Score: 0.09545322507619858


In [87]:
document = [
    "Transformers are power models",
    "FAISS is used for efficient similarity search",
    "Sentence transformers provide sentence embeddings",
    "BERT is a bidirectional trasnformer model."
]

In [88]:
# encode documents

doc_embeddings = model.encode(document)

#Build Fiass index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

In [89]:
# query the system

In [90]:
query = "What models are used for NLP?"

query_embedding = model.encode([query])

In [91]:
k = 3 #top results

In [92]:
distances, indices = index.search(np.array(query_embedding), k)

In [93]:
print("Query: ", query)
print("\n Top Results: \n")

Query:  What models are used for NLP?

 Top Results: 



In [94]:
# how to implement a rag`

for i, idx in enumerate(indices[0]):
  print(f"{i+1}. {document[idx]}")
  print("Distance: ", distances[0][i])


1. Sentence transformers provide sentence embeddings
Distance:  0.9839968
2. BERT is a bidirectional trasnformer model.
Distance:  1.0919342
3. FAISS is used for efficient similarity search
Distance:  1.3759673


In [95]:
# how to implement a rag`

Class activity
sentence - The bank is near the river

Ask: what does "bank" mean?
how does biderectinality resolve ambiguity?

In [96]:
document_activity = [
   "The bank is near the river.",
   "The cat is near the bank.",
   "Deposit 10000 RS in the bank.",
   "Deposit 10000 RS in the bank near river.",
   "Deposit 10000 RS in the bank near river bank.",
]

In [97]:
# encode documents

doc_embeddings_activity = model.encode(document_activity)


In [98]:
doc_embeddings_activity

array([[ 0.04531943,  0.04188718, -0.09180358, ..., -0.00922927,
         0.04889701, -0.03473015],
       [ 0.11185404,  0.02337263, -0.04520846, ...,  0.04493994,
         0.05220203, -0.00056371],
       [ 0.04776569,  0.04713394, -0.08673548, ...,  0.00899314,
        -0.02961051, -0.07171874],
       [ 0.03421022,  0.06116208, -0.09238008, ...,  0.01159026,
        -0.02536817, -0.05910772],
       [ 0.03552124,  0.05955334, -0.09952907, ...,  0.01969789,
        -0.01616849, -0.06143783]], dtype=float32)

In [99]:
doc_embeddings_activity.shape

(5, 384)

In [100]:
#Build Fiass index
dimension_activity = doc_embeddings_activity.shape[1]
index_activity = faiss.IndexFlatL2(dimension_activity)
index_activity.add(np.array(doc_embeddings_activity))

In [101]:
query_activity = "What does bank mean?"

query_embedding_activity = model.encode([query])

In [102]:
distances_act, indices_act = index.search(np.array(query_embedding_activity), k=5)

In [103]:
for i, idx in enumerate(indices_act[0]):
  print(f"{i+1}. {document_activity[idx]}")
  print("Distance: ", distances_act[0][i])


1. Deposit 10000 RS in the bank.
Distance:  0.9839968
2. Deposit 10000 RS in the bank near river.
Distance:  1.0919342
3. The cat is near the bank.
Distance:  1.3759673
4. The bank is near the river.
Distance:  1.5070686
5. Deposit 10000 RS in the bank near river bank.
Distance:  3.4028235e+38


Using bert model for document_activity

In [104]:
from transformers import BertTokenizer

In [105]:
tokenizer = BertTokenizer.from_pretrained("bert-base-cased")

# Understanding Tokenizer Outputs: `input_ids` and `attention_mask`

input_ids and attention_mask are indeed part of the output when you use a tokenizer (like the BertTokenizer) to process text, especially when you specify `return_tensors='np'` (for NumPy arrays) or `return_tensors='pt'` (for PyTorch tensors). They are essential components that the BERT model uses to understand the input:

*   **`input_ids`**: These are the numerical representations of your text, where each number corresponds to a specific token (word, subword, or special token) in the tokenizer's vocabulary.
*   **`attention_mask`**: This is a binary mask that tells the model which tokens are actual content and which are padding. A `1` indicates a real token that the model should attend to, and a `0` indicates a padding token that should be ignored during computation. This ensures that the model doesn't try to learn patterns from the padding.

In [106]:
encoded_inputs = tokenizer(document_activity, padding=True, truncation=True, return_tensors='np')
print("Token IDs (input_ids):")
print(encoded_inputs['input_ids'])
print("Attention Mask:")
print(encoded_inputs['attention_mask'])
print("Token IDs shape:", encoded_inputs['input_ids'].shape)

Token IDs (input_ids):
[[  101  1109  3085  1110  1485  1103  2186   119   102     0     0     0
      0     0     0     0]
 [  101  1109  5855  1110  1485  1103  3085   119   102     0     0     0
      0     0     0     0]
 [  101  3177  5674  5053  1204  6087  1568 25591  1107  1103  3085   119
    102     0     0     0]
 [  101  3177  5674  5053  1204  6087  1568 25591  1107  1103  3085  1485
   2186   119   102     0]
 [  101  3177  5674  5053  1204  6087  1568 25591  1107  1103  3085  1485
   2186  3085   119   102]]
Attention Mask:
[[1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0]
 [1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]
Token IDs shape: (5, 16)


# Understanding BERT Tokenization Output

When using `tokenizer.encode()` with a list of sentences, the method returns a list of lists, where each inner list contains the token IDs for a specific sentence.

## Why a `ValueError` for `np.array(encoding_activity)`?

The `ValueError` you encountered when trying to convert `encoding_activity` directly to a NumPy array (e.g., `np.array(encoding_activity)`) happened because the tokenized sentences had **different lengths**. NumPy cannot form a rectangular (homogeneous) array from a list of lists with varying lengths. For instance, 'The bank is near the river.' might have a different number of tokens than 'Deposit 10000 RS in the bank near river bank.'.

## Handling Variable-Length Sequences

To properly handle variable-length sequences with BERT tokenization, you typically need to:

*   **Pad shorter sequences:** Make all sequences the same length.
*   **Truncate longer sequences:** Shorten sequences that exceed a maximum length.

This is usually achieved by passing `padding=True` and `truncation=True` to the tokenizer's `encode_plus` or `__call__` methods. You can also specify `return_tensors='np'` or `return_tensors='pt'` to directly get NumPy arrays or PyTorch tensors.

## Explanation of the Tokenized Output

After applying padding and truncation, the code successfully tokenized the `document_activity` list using the BERT tokenizer. Here's what the output `encoded_inputs` means:

*   **`Token IDs (input_ids)`**: This is a 2D NumPy array. Each row corresponds to one of your input sentences, and the numbers within each row are the numerical representations (token IDs) of the words in that sentence. Special tokens are included:
    *   `101`: For `[CLS]`, indicating the start of a sequence.
    *   `102`: For `[SEP]`, indicating the end of a sequence.
    *   `0`: Represents **padding tokens**, added to make all sequences the same length.

*   **`Attention Mask`**: This is also a 2D NumPy array, with the same shape as `input_ids`. It tells the model which tokens to pay attention to:
    *   `1`: Indicates an actual word token from your sentence.
    *   `0`: Indicates a padding token, telling the model to ignore it during processing.

*   **`Token IDs shape: (5, 16)`**: This indicates:
    *   You have **5** tokenized sequences (corresponding to your 5 sentences in `document_activity`).
    *   Each sequence has been padded or truncated to a maximum length of **16** tokens.

### Creating Sentence Embeddings using `BertModel`

While `sentence-transformers` simplifies getting sentence embeddings, you can also use the raw `BertModel` from the `transformers` library. A common approach is to take the output embedding of the special `[CLS]` token as the sentence representation.

In [107]:
from transformers import BertModel
import torch

# Load the pre-trained BERT model
bert_model = BertModel.from_pretrained('bert-base-cased')

# Put the model in evaluation mode
bert_model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

Now, we will tokenize the `document_activity` list and pass it through the `bert_model`. We'll extract the `[CLS]` token's embedding as our sentence embedding.

In [108]:
# Tokenize the sentences from document_activity
# 'encoded_inputs' was already created in a previous cell using the same tokenizer

# Convert numpy arrays to PyTorch tensors for the BertModel
input_ids = torch.tensor(encoded_inputs['input_ids'])
attention_mask = torch.tensor(encoded_inputs['attention_mask'])

# Pass the tokenized input through the BERT model
with torch.no_grad(): # Disable gradient calculations for inference
    outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)

# The last hidden state contains the embeddings for all tokens
last_hidden_states = outputs.last_hidden_state

# To get sentence embeddings, we typically take the embedding of the [CLS] token
# The [CLS] token is always the first token (index 0) of each sequence
cls_embeddings = last_hidden_states[:, 0, :]

print("Shape of BERT sentence embeddings (from [CLS] token):")
print(cls_embeddings.shape)
print("\nFirst 5 values of the first sentence embedding:")
print(cls_embeddings[0, :5])

Shape of BERT sentence embeddings (from [CLS] token):
torch.Size([5, 768])

First 5 values of the first sentence embedding:
tensor([ 0.3358,  0.0873, -0.0614, -0.2334, -0.1395])


The `cls_embeddings` tensor now holds a sentence embedding for each sentence in `document_activity`, where each embedding is a 768-dimensional vector (for `bert-base-cased`).

In [109]:
print("All BERT sentence embeddings (from [CLS] token):")
for i, embedding in enumerate(cls_embeddings):
    print(f"Sentence {i+1} Embedding: {embedding.tolist()}")

All BERT sentence embeddings (from [CLS] token):
Sentence 1 Embedding: [0.3357871174812317, 0.08734066784381866, -0.0614219531416893, -0.23341414332389832, -0.1395442634820938, 0.21118107438087463, 0.20812013745307922, -0.17000944912433624, 0.1527717411518097, -1.2129919528961182, -0.48069846630096436, -0.014834457077085972, -0.19742143154144287, -0.05685271695256233, -0.4415462911128998, 0.025674352422356606, 0.18839824199676514, 0.07811692357063293, -0.2729480266571045, 0.041455283761024475, 0.08037388324737549, -0.2812661826610565, 0.6537659168243408, -0.33783459663391113, 0.2435086965560913, 0.005316141992807388, 0.48500534892082214, 0.08971653133630753, 0.08166884630918503, 0.20402027666568756, 0.15493543446063995, 0.23934651911258698, -0.3303915858268738, -0.4092681109905243, -0.14430783689022064, 0.5885985493659973, -0.05143679305911064, -0.4134417772293091, 0.2046785056591034, -0.13015617430210114, -0.48242026567459106, 0.013326841406524181, 0.7474206686019897, 0.24263666570186

In [110]:
cls_embeddings.shape

torch.Size([5, 768])

In [111]:
import torch.nn.functional as F

# Calculate pairwise cosine similarity
# The cls_embeddings are already a torch.Tensor
bert_cosine_scores = F.cosine_similarity(cls_embeddings.unsqueeze(1), cls_embeddings.unsqueeze(0), dim=-1)

print("Pairwise Cosine Similarities (BERT embeddings):")
print(bert_cosine_scores)

Pairwise Cosine Similarities (BERT embeddings):
tensor([[1.0000, 0.9807, 0.9476, 0.9613, 0.9585],
        [0.9807, 1.0000, 0.9498, 0.9561, 0.9562],
        [0.9476, 0.9498, 1.0000, 0.9839, 0.9892],
        [0.9613, 0.9561, 0.9839, 1.0000, 0.9976],
        [0.9585, 0.9562, 0.9892, 0.9976, 1.0000]])


In [112]:
print("Interpreting BERT Cosine Similarities:")
for i in range(len(document_activity)):
  for j in range(i+1, len(document_activity)):
    print(f"'{document_activity[i]}' \t vs \t '{document_activity[j]}' \t Score: {bert_cosine_scores[i][j]:.4f}")

Interpreting BERT Cosine Similarities:
'The bank is near the river.' 	 vs 	 'The cat is near the bank.' 	 Score: 0.9807
'The bank is near the river.' 	 vs 	 'Deposit 10000 RS in the bank.' 	 Score: 0.9476
'The bank is near the river.' 	 vs 	 'Deposit 10000 RS in the bank near river.' 	 Score: 0.9613
'The bank is near the river.' 	 vs 	 'Deposit 10000 RS in the bank near river bank.' 	 Score: 0.9585
'The cat is near the bank.' 	 vs 	 'Deposit 10000 RS in the bank.' 	 Score: 0.9498
'The cat is near the bank.' 	 vs 	 'Deposit 10000 RS in the bank near river.' 	 Score: 0.9561
'The cat is near the bank.' 	 vs 	 'Deposit 10000 RS in the bank near river bank.' 	 Score: 0.9562
'Deposit 10000 RS in the bank.' 	 vs 	 'Deposit 10000 RS in the bank near river.' 	 Score: 0.9839
'Deposit 10000 RS in the bank.' 	 vs 	 'Deposit 10000 RS in the bank near river bank.' 	 Score: 0.9892
'Deposit 10000 RS in the bank near river.' 	 vs 	 'Deposit 10000 RS in the bank near river bank.' 	 Score: 0.9976


### Querying with BERT Embeddings

In [113]:
# Define the query string
bert_query_text = "What does bank mean?"

# Tokenize the query using the BertTokenizer
encoded_bert_query = tokenizer(bert_query_text, return_tensors='pt', padding=True, truncation=True)

# Pass the tokenized query through the BertModel to get embeddings
with torch.no_grad():
    bert_query_outputs = bert_model(input_ids=encoded_bert_query['input_ids'], attention_mask=encoded_bert_query['attention_mask'])

# Extract the [CLS] token embedding for the query
bert_query_embedding = bert_query_outputs.last_hidden_state[:, 0, :]

# Convert to numpy for FAISS or further processing if needed
bert_query_embedding_np = bert_query_embedding.cpu().numpy()

print("Shape of BERT query embedding:", bert_query_embedding_np.shape)
# print("First 5 values of BERT query embedding:", bert_query_embedding_np[:, :])

Shape of BERT query embedding: (1, 768)


### Performing Similarity Search with BERT Embeddings and FAISS

In [114]:
# Convert BERT sentence embeddings to NumPy and ensure C-contiguous for FAISS
bert_doc_embeddings_np = np.ascontiguousarray(cls_embeddings.cpu().numpy())

# Build FAISS index for BERT embeddings
bert_dimension = bert_doc_embeddings_np.shape[1]
bert_index = faiss.IndexFlatL2(bert_dimension)
bert_index.add(bert_doc_embeddings_np)

print(f"FAISS index for BERT embeddings created with dimension: {bert_dimension}")

FAISS index for BERT embeddings created with dimension: 768


In [115]:
# Query the BERT FAISS index with the BERT query embedding
k_bert = len(document_activity) # Get all results for comparison
bert_distances, bert_indices = bert_index.search(bert_query_embedding_np, k_bert)

print("Query: ", bert_query_text)
print("\nTop Results (BERT embeddings): \n")
for i, idx in enumerate(bert_indices[0]):
    print(f"{i+1}. {document_activity[idx]}")
    print(f"   Distance: {bert_distances[0][i]:.4f}")

Query:  What does bank mean?

Top Results (BERT embeddings): 

1. The cat is near the bank.
   Distance: 29.3217
2. The bank is near the river.
   Distance: 39.5177
3. Deposit 10000 RS in the bank.
   Distance: 48.2487
4. Deposit 10000 RS in the bank near river.
   Distance: 49.3587
5. Deposit 10000 RS in the bank near river bank.
   Distance: 49.6946


### Generating Word Embeddings for Each Word in `document_activity` using BERT

In [116]:
import torch

# Tokenize the document_activity to get input_ids and attention_mask
# The previous encoded_inputs already contains this, but for clarity, re-tokenize
# to ensure we have the necessary token information for each word.
# Using return_tensors='pt' for direct use with BertModel.
word_tokenized_inputs = tokenizer(document_activity, padding=True, truncation=True, return_tensors='pt')

# Get input_ids and attention_mask
input_ids_words = word_tokenized_inputs['input_ids']
attention_mask_words = word_tokenized_inputs['attention_mask']

print("Tokenized input_ids shape:", input_ids_words.shape)
print("Attention mask shape:", attention_mask_words.shape)

Tokenized input_ids shape: torch.Size([5, 16])
Attention mask shape: torch.Size([5, 16])


In [117]:
# Pass the tokenized inputs through the BERT model to get word embeddings
# Ensure the model is in evaluation mode and disable gradient calculation
bert_model.eval()
with torch.no_grad():
    word_outputs = bert_model(input_ids=input_ids_words, attention_mask=attention_mask_words)

# The last_hidden_state contains the embeddings for all tokens in each sequence
word_embeddings = word_outputs.last_hidden_state

print("Shape of BERT word embeddings:", word_embeddings.shape)

print("\n--- Word Embeddings for document_activity ---")
for i, sentence_input_ids in enumerate(input_ids_words):
    print(f"\nSentence {i+1}: '{document_activity[i]}'")
    # Iterate through tokens in the current sentence (up to its actual length using attention_mask)
    for j in range(len(sentence_input_ids)):
        if attention_mask_words[i][j] == 1: # Only process actual tokens, not padding
            token_id = sentence_input_ids[j].item()
            token = tokenizer.convert_ids_to_tokens(token_id)
            embedding = word_embeddings[i][j]
            print(f"  Token: '{token}' (ID: {token_id}) - Embedding snippet: {embedding[:5].tolist()}")


Shape of BERT word embeddings: torch.Size([5, 16, 768])

--- Word Embeddings for document_activity ---

Sentence 1: 'The bank is near the river.'
  Token: '[CLS]' (ID: 101) - Embedding snippet: [0.3357871174812317, 0.08734066784381866, -0.0614219531416893, -0.23341414332389832, -0.1395442634820938]
  Token: 'The' (ID: 1109) - Embedding snippet: [0.3948969542980194, -0.5667884945869446, 0.419200599193573, -0.2993219792842865, -0.23533234000205994]
  Token: 'bank' (ID: 3085) - Embedding snippet: [0.1907953917980194, 0.3835418224334717, -0.37098750472068787, 0.2599014639854431, 0.30933263897895813]
  Token: 'is' (ID: 1110) - Embedding snippet: [-0.009286384098231792, -0.03313722088932991, 0.10600897669792175, 0.38592684268951416, 0.4812963604927063]
  Token: 'near' (ID: 1485) - Embedding snippet: [0.5583838224411011, 0.022685175761580467, -0.34568196535110474, 0.27321165800094604, 0.17120270431041718]
  Token: 'the' (ID: 1103) - Embedding snippet: [0.09700772166252136, -0.4058184921741485

In [118]:
import torch.nn.functional as F

# Find the token ID for 'bank'
bank_token_id = tokenizer.convert_tokens_to_ids('bank')

bank_embeddings = []
bank_contexts = []

# Extract all embeddings for the token 'bank'
for i, sentence_input_ids in enumerate(input_ids_words):
    for j in range(len(sentence_input_ids)):
        if attention_mask_words[i][j] == 1 and sentence_input_ids[j].item() == bank_token_id:
            bank_embeddings.append(word_embeddings[i][j])
            bank_contexts.append(f"'{document_activity[i]}' (pos {j})")

if len(bank_embeddings) > 1:
    # Convert list of embeddings to a tensor
    bank_embeddings_tensor = torch.stack(bank_embeddings)

    # Calculate pairwise cosine similarity
    # unsqueeze for broadcasting: [N, D] -> [N, 1, D] and [1, N, D]
    cosine_similarities_bank = F.cosine_similarity(bank_embeddings_tensor.unsqueeze(1), bank_embeddings_tensor.unsqueeze(0), dim=-1)

    print("\n--- Cosine Similarities for 'bank' embeddings ---")
    for i in range(len(bank_contexts)):
        for j in range(i + 1, len(bank_contexts)):
            print(f"Similarity between {bank_contexts[i]} and {bank_contexts[j]}: {cosine_similarities_bank[i][j]:.4f}")
else:
    print("Not enough 'bank' occurrences to calculate pairwise similarity.")


--- Cosine Similarities for 'bank' embeddings ---
Similarity between 'The bank is near the river.' (pos 2) and 'The cat is near the bank.' (pos 6): 0.9293
Similarity between 'The bank is near the river.' (pos 2) and 'Deposit 10000 RS in the bank.' (pos 10): 0.7934
Similarity between 'The bank is near the river.' (pos 2) and 'Deposit 10000 RS in the bank near river.' (pos 10): 0.8530
Similarity between 'The bank is near the river.' (pos 2) and 'Deposit 10000 RS in the bank near river bank.' (pos 10): 0.8444
Similarity between 'The bank is near the river.' (pos 2) and 'Deposit 10000 RS in the bank near river bank.' (pos 13): 0.7898
Similarity between 'The cat is near the bank.' (pos 6) and 'Deposit 10000 RS in the bank.' (pos 10): 0.8409
Similarity between 'The cat is near the bank.' (pos 6) and 'Deposit 10000 RS in the bank near river.' (pos 10): 0.8370
Similarity between 'The cat is near the bank.' (pos 6) and 'Deposit 10000 RS in the bank near river bank.' (pos 10): 0.8337
Similarity

The code successfully calculated the pairwise cosine similarities for all occurrences of the word 'bank' within your `document_activity` list, using the BERT word embeddings. You can observe how the similarity scores reflect the context in which 'bank' is used:

*   **High Similarity** (e.g., `0.9293`, `0.9865`): Occurrences of 'bank' that refer to the same concept (like 'river bank' or 'financial institution') tend to have higher cosine similarity scores.
    *   For example, `'The bank is near the river.'` and `'The cat is near the bank.'` have a similarity of `0.9293`, suggesting both refer to the geographical bank.
    *   Similarly, `'Deposit 10000 RS in the bank near river.'` and `'Deposit 10000 RS in the bank near river bank.'` have a very high similarity of `0.9865`, indicating both refer to the financial institution, even though one includes 'river bank' which could be ambiguous.

*   **Lower Similarity** (e.g., `0.7934`, `0.6795`): Occurrences where 'bank' carries a different meaning show relatively lower similarities.
    *   For instance, comparing `'The bank is near the river.'` (geographical bank) with `'Deposit 10000 RS in the bank.'` (financial institution) yields a similarity of `0.7934`, which is lower than the similarities between two 'river bank' contexts.

This demonstrates BERT's ability to generate contextualized word embeddings, meaning the embedding for a word changes based on its surrounding words, allowing it to differentiate between different senses of polysemous words like 'bank'.

In [121]:
# Ensure query_bank_embedding and bank_embeddings_tensor are available from previous steps

# Convert 'bank' document embeddings to NumPy and ensure C-contiguous for FAISS
# (bank_embeddings_tensor was created in the previous cell if 'bank' was found)
if 'bank_embeddings_tensor' in locals() and bank_embeddings_tensor is not None:
    bank_doc_word_embeddings_np = np.ascontiguousarray(bank_embeddings_tensor.cpu().numpy())

    # Build FAISS index for 'bank' word embeddings
    bank_word_dimension = bank_doc_word_embeddings_np.shape[1]
    bank_word_index = faiss.IndexFlatL2(bank_word_dimension)
    bank_word_index.add(bank_doc_word_embeddings_np)

    print(f"FAISS index for 'bank' word embeddings created with dimension: {bank_word_dimension}")

    # Convert query_bank_embedding to numpy for FAISS search
    if query_bank_embedding is not None:
        query_bank_embedding_np = query_bank_embedding.cpu().numpy().reshape(1, -1)

        # Query the FAISS index with the 'bank' word embedding from the query
        k_bank_word = len(bank_embeddings) # Get all results for comparison
        bank_word_distances, bank_word_indices = bank_word_index.search(query_bank_embedding_np, k_bank_word)

        print(f"\nQuery: {query_bank_context}")
        print("\nTop Results (word embeddings for 'bank'):\n")
        for i, idx in enumerate(bank_word_indices[0]):
            print(f"{i+1}. {bank_contexts[idx]}")
            print(f"   Distance: {bank_word_distances[0][i]:.4f}")
    else:
        print("Query 'bank' embedding not found. Cannot perform search.")
else:
    print("'bank' word embeddings from documents not found. Cannot create index.")

FAISS index for 'bank' word embeddings created with dimension: 768

Query: Query: 'What does bank mean?' (pos 3)

Top Results (word embeddings for 'bank'):

1. 'The cat is near the bank.' (pos 6)
   Distance: 99.4995
2. 'Deposit 10000 RS in the bank.' (pos 10)
   Distance: 106.9693
3. 'The bank is near the river.' (pos 2)
   Distance: 110.3709
4. 'Deposit 10000 RS in the bank near river bank.' (pos 10)
   Distance: 157.3094
5. 'Deposit 10000 RS in the bank near river.' (pos 10)
   Distance: 161.1176
6. 'Deposit 10000 RS in the bank near river bank.' (pos 13)
   Distance: 194.8431


While you have 5 sentences in your `document_activity` list, the last sentence, `'Deposit 10000 RS in the bank near river bank.'`, contains the word 'bank' twice.

When we extracted the word embeddings for 'bank', we captured each instance of the word. Therefore, `bank_embeddings` contains 6 distinct word embeddings for 'bank' across your 5 sentences:

*   `'The bank'` (from sentence 1)
*   `'the bank'` (from sentence 2)
*   `'the bank'` (from sentence 3)
*   `'the bank'` (from sentence 4)
*   `'the bank'` (first instance from sentence 5)
*   `'river bank'` (second instance from sentence 5)

So, the FAISS search correctly returns 6 results because it's searching against 6 individual word embeddings for 'bank'.

### Using SentenceTransformer's underlying model for Word Embeddings

To get word embeddings from a `SentenceTransformer` model, you need to access the underlying `transformers` model and then pass your tokenized input through it. The `SentenceTransformer` model is typically composed of a `transformers` model followed by a pooling layer. We'll use the `tokenizer` and `model` (from `SentenceTransformer`) that are already loaded.

In [125]:
# Access the underlying Hugging Face Transformers model from SentenceTransformer
# The base transformer model is usually the first module (index 0) in the pipeline
base_transformer_model = model._modules['0'].model
base_transformer_tokenizer = model._modules['0'].tokenizer

# Ensure the base model is in evaluation mode
base_transformer_model.eval()

# Example sentence for word embedding extraction
example_sentence = "The bank is near the river."

# Tokenize the example sentence using the base tokenizer
encoded_example_sentence = base_transformer_tokenizer(
    example_sentence,
    return_tensors='pt',
    padding=True,
    truncation=True
)

# Extract input_ids and attention_mask
input_ids = encoded_example_sentence['input_ids']
attention_mask = encoded_example_sentence['attention_mask']

# Get word embeddings from the base transformer model
with torch.no_grad():
    outputs = base_transformer_model(input_ids=input_ids, attention_mask=attention_mask)

# The last_hidden_state contains the word embeddings for all tokens
word_embeddings_from_st = outputs.last_hidden_state

print("Shape of word embeddings from SentenceTransformer's base model:", word_embeddings_from_st.shape)

print("\n--- Word Embeddings for example sentence ---")
tokens = base_transformer_tokenizer.convert_ids_to_tokens(input_ids[0])
for i, token in enumerate(tokens):
    if attention_mask[0][i] == 1: # Only print for actual tokens, not padding
        embedding = word_embeddings_from_st[0][i]
        print(f"  Token: '{token}' - Embedding snippet: {embedding[:5].tolist()}")

Shape of word embeddings from SentenceTransformer's base model: torch.Size([1, 9, 384])

--- Word Embeddings for example sentence ---
  Token: '[CLS]' - Embedding snippet: [0.18353964388370514, 0.09942851215600967, -0.42496615648269653, -0.041904740035533905, -0.2947656810283661]
  Token: 'the' - Embedding snippet: [0.050382692366838455, 0.3006901741027832, -0.5701708197593689, 0.18330614268779755, -0.07104481011629105]
  Token: 'bank' - Embedding snippet: [0.6166223883628845, 0.3429644703865051, -0.939159631729126, -0.6539825201034546, -0.5061778426170349]
  Token: 'is' - Embedding snippet: [0.21782729029655457, 0.1835787296295166, -0.5027193427085876, 0.14710037410259247, -0.0019047120586037636]
  Token: 'near' - Embedding snippet: [0.1505209505558014, 0.49277207255363464, -0.3566091060638428, -0.17062397301197052, 0.3463687598705292]
  Token: 'the' - Embedding snippet: [0.08987230807542801, 0.2378261238336563, -0.5504907369613647, 0.16012534499168396, -0.011247274465858936]
  Token:

In [126]:
import torch.nn.functional as F

# Reshape word_embeddings_from_st to be [num_tokens, embedding_dim]
# We currently have [1, num_tokens, embedding_dim] due to batching a single sentence
word_embeddings_single_sentence = word_embeddings_from_st.squeeze(0)

# Calculate pairwise cosine similarity
# unsqueeze for broadcasting: [N, D] -> [N, 1, D] and [1, N, D]
cosine_similarities_words_st = F.cosine_similarity(word_embeddings_single_sentence.unsqueeze(1), word_embeddings_single_sentence.unsqueeze(0), dim=-1)

print("Pairwise Cosine Similarities (Word embeddings from SentenceTransformer):")
print(cosine_similarities_words_st)

print("\nInterpreting Word Cosine Similarities from SentenceTransformer:")
# Get the tokens for interpretation
tokens = base_transformer_tokenizer.convert_ids_to_tokens(input_ids[0])
actual_tokens = [token for i, token in enumerate(tokens) if attention_mask[0][i] == 1]

for i in range(len(actual_tokens)):
  for j in range(i + 1, len(actual_tokens)):
    print(f"'{actual_tokens[i]}' \t vs \t '{actual_tokens[j]}' \t Score: {cosine_similarities_words_st[i][j]:.4f}")

Pairwise Cosine Similarities (Word embeddings from SentenceTransformer):
tensor([[1.0000, 0.4996, 0.4926, 0.4026, 0.2109, 0.5227, 0.3716, 0.5224, 0.5316],
        [0.4996, 1.0000, 0.5917, 0.8367, 0.4896, 0.9470, 0.5175, 0.9058, 0.9378],
        [0.4926, 0.5917, 1.0000, 0.4825, 0.2806, 0.5981, 0.5605, 0.5694, 0.6275],
        [0.4026, 0.8367, 0.4825, 1.0000, 0.6214, 0.7802, 0.4553, 0.8509, 0.8274],
        [0.2109, 0.4896, 0.2806, 0.6214, 1.0000, 0.5085, 0.3260, 0.4682, 0.4704],
        [0.5227, 0.9470, 0.5981, 0.7802, 0.5085, 1.0000, 0.5867, 0.8741, 0.9260],
        [0.3716, 0.5175, 0.5605, 0.4553, 0.3260, 0.5867, 1.0000, 0.4990, 0.5491],
        [0.5224, 0.9058, 0.5694, 0.8509, 0.4682, 0.8741, 0.4990, 1.0000, 0.9573],
        [0.5316, 0.9378, 0.6275, 0.8274, 0.4704, 0.9260, 0.5491, 0.9573, 1.0000]])

Interpreting Word Cosine Similarities from SentenceTransformer:
'[CLS]' 	 vs 	 'the' 	 Score: 0.4996
'[CLS]' 	 vs 	 'bank' 	 Score: 0.4926
'[CLS]' 	 vs 	 'is' 	 Score: 0.4026
'[CLS]' 	 vs

### Querying with Word Embeddings (from SentenceTransformer's base model)

In [129]:
import torch.nn.functional as F

# Define the query string
query_word_text = "What does bank mean?"

# Tokenize the query using the base tokenizer from SentenceTransformer
encoded_query_word = base_transformer_tokenizer(
    query_word_text,
    return_tensors='pt',
    padding=True,
    truncation=True
)

# Pass the tokenized query through the base transformer model
with torch.no_grad():
    query_word_outputs = base_transformer_model(
        input_ids=encoded_query_word['input_ids'],
        attention_mask=encoded_query_word['attention_mask']
    )

# Extract the embedding for the word 'bank' from the query
query_tokens = base_transformer_tokenizer.convert_ids_to_tokens(encoded_query_word['input_ids'][0])

# Find the position of 'bank' in the query tokens
try:
    bank_query_pos = query_tokens.index('bank')
    query_bank_embedding = query_word_outputs.last_hidden_state[:, bank_query_pos, :].squeeze(0)
    query_bank_context = f"'{query_word_text}' (pos {bank_query_pos})" # Modified line
    print("Shape of query 'bank' embedding:", query_bank_embedding.shape)
    print("First 5 values of query 'bank' embedding:", query_bank_embedding[:5].tolist())
except ValueError:
    query_bank_embedding = None
    query_bank_context = "'bank' not found in query."
    print("The word 'bank' was not found in the query string.")

# --- BEGIN MODIFICATION ---
# Regenerate document bank embeddings using the base_transformer_model (384-dim) for consistency

# Tokenize document_activity using base_transformer_tokenizer
word_tokenized_docs_st = base_transformer_tokenizer(document_activity, padding=True, truncation=True, return_tensors='pt')

# Pass through base_transformer_model to get word embeddings
with torch.no_grad():
    word_outputs_st_docs = base_transformer_model(
        input_ids=word_tokenized_docs_st['input_ids'],
        attention_mask=word_tokenized_docs_st['attention_mask']
    )
word_embeddings_st_docs = word_outputs_st_docs.last_hidden_state

# Find the token ID for 'bank' using the base_transformer_tokenizer
bank_token_id_st = base_transformer_tokenizer.convert_tokens_to_ids('bank')

bank_embeddings_st = [] # This will store the 384-dim embeddings
bank_contexts_st = [] # This will store the contexts for the 384-dim embeddings

# Extract all embeddings for the token 'bank' from the document_activity
for i, sentence_input_ids in enumerate(word_tokenized_docs_st['input_ids']):
    for j in range(len(sentence_input_ids)):
        if word_tokenized_docs_st['attention_mask'][i][j] == 1 and sentence_input_ids[j].item() == bank_token_id_st:
            bank_embeddings_st.append(word_embeddings_st_docs[i][j])
            bank_contexts_st.append(f"'{document_activity[i]}' (pos {j})")

# Combine the query 'bank' embedding with the newly generated document 'bank' embeddings
all_bank_embeddings = []
all_bank_contexts = []

if query_bank_embedding is not None:
    all_bank_embeddings.append(query_bank_embedding)
    all_bank_contexts.append(query_bank_context)

# Add the 384-dim document bank embeddings
if bank_embeddings_st:
    all_bank_embeddings.extend(bank_embeddings_st)
    all_bank_contexts.extend(bank_contexts_st)

# --- END MODIFICATION ---

if len(all_bank_embeddings) > 1:
    all_bank_embeddings_tensor = torch.stack(all_bank_embeddings)
    # Convert to numpy for FAISS and ensure C-contiguous
    all_bank_doc_word_embeddings_np = np.ascontiguousarray(all_bank_embeddings_tensor.cpu().numpy())

    # Build FAISS index for all 'bank' word embeddings (query + documents)
    all_bank_word_dimension = all_bank_doc_word_embeddings_np.shape[1]
    all_bank_word_index = faiss.IndexFlatL2(all_bank_word_dimension)
    all_bank_word_index.add(all_bank_doc_word_embeddings_np)

    print(f"FAISS index for all 'bank' word embeddings created with dimension: {all_bank_word_dimension}")

    # Query the FAISS index with the query 'bank' embedding
    k_all_bank_word = len(all_bank_contexts) # Get all results for comparison
    all_bank_word_distances, all_bank_word_indices = all_bank_word_index.search(query_bank_embedding.cpu().numpy().reshape(1, -1), k_all_bank_word)

    print(f"\nQuery: {query_bank_context}")
    print("\nTop Results (word embeddings for 'bank'):\n")
    # Adjusted loop to ensure '1.' is printed for the top result, even if it's not the query itself
    current_rank = 1
    for i, idx in enumerate(all_bank_word_indices[0]):
        # Skip the query itself if it appears as the top result (distance 0)
        if idx == 0 and all_bank_word_distances[0][i] < 1e-6: # Check for near-zero distance
            continue
        print(f"{current_rank}. {all_bank_contexts[idx]}")
        print(f"   Distance: {all_bank_word_distances[0][i]:.4f}")
        current_rank += 1
else:
    print("Not enough 'bank' embeddings (query or documents) to perform a meaningful search.")

Shape of query 'bank' embedding: torch.Size([384])
First 5 values of query 'bank' embedding: [0.3250919282436371, 0.4827585220336914, -1.1539138555526733, -0.761574387550354, -0.7423349618911743]
FAISS index for all 'bank' word embeddings created with dimension: 384

Query: 'What does bank mean?' (pos 3)

Top Results (word embeddings for 'bank'):

1. 'The bank is near the river.' (pos 2)
   Distance: 16.0136
2. 'The cat is near the bank.' (pos 6)
   Distance: 28.5982
3. 'Deposit 10000 RS in the bank near river.' (pos 7)
   Distance: 32.6376
4. 'Deposit 10000 RS in the bank near river bank.' (pos 7)
   Distance: 41.9448
5. 'Deposit 10000 RS in the bank.' (pos 7)
   Distance: 56.2238
6. 'Deposit 10000 RS in the bank near river bank.' (pos 10)
   Distance: 60.9257
